In [1]:
from Library.Build_Dataset import *
DIRECTORY = "../../"

def printout(V, Stats, model): 
    # printing Stats
    print("R2 = %.2f (+/- %.2f) Constraint = %.2f (+/- %.2f)" % \
          (Stats.train_objective[0], Stats.train_objective[1],
           Stats.train_loss[0], Stats.train_loss[1]))
    Vout = tf.convert_to_tensor(np.float32(model.Y))
    Loss_norm, dLoss = Loss_Vout(V, model.Pout, Vout)
    print('Loss Targets', np.mean(Loss_norm))
    Loss_norm, dLoss = Loss_SV(V, model.S)
    print('Loss SV', np.mean(Loss_norm))
    Vin = tf.convert_to_tensor(np.float32(model.X))
    Pin = tf.convert_to_tensor(np.float32(model.Pin))
    if Vin.shape[1] == model.S.shape[1]: # special case
        Vin  = tf.linalg.matmul(Vin, tf.transpose(Pin), b_is_sparse=True)
    Loss_norm, dLoss = Loss_Vin(V, model.Pin, Vin, model.mediumbound,model)
    print('Loss Vin bound', np.mean(Loss_norm))
    Loss_norm, dLoss = Loss_Vpos(V, model)
    print('Loss V positive', np.mean(Loss_norm))

In [4]:
# Generate training set with E coli iML1515 with FBA simulation 
# constrained by experimental file: metabolites in medium are not drawn at
# random but are the same than in the provided training experimental file
# This cell may take several hours to execute! Avoid running this in Colab
    
# What you can change
seed = 10
np.random.seed(seed=seed)  # seed for random number generator
cobraname =  'iML1515_duplicated' # name of the model 
mediumname = 'iML1515' # name of the medium file 
mediumbound = 'UB' # Exact bound (EB) or upper bound (UB)
expname = 'iML1515_EXP' # name of the experimental dataset for constraints
method = 'pFBA' # FBA, pFBA or EXP
size, size_i  = 110, 10 # expname training set size, training set size per item in expname
reduce = True # Set at True if you want to reduce the model
verbose = True
# End of What you can change

# Get X from experimental data set
cobrafile = DIRECTORY+'Dataset_input/'+cobraname
expfile  = DIRECTORY+'Dataset_input/'+expname
parameter = TrainingSet(cobraname=cobrafile, 
                        mediumname=expfile, 
                        mediumbound=mediumbound, 
                        mediumsize=38, 
                        method='EXP',verbose=False)
X = parameter.X.copy()

# Get other parameters from medium file
mediumfile = DIRECTORY+'Dataset_input/'+mediumname
parameter = TrainingSet(cobraname=cobrafile, 
                        mediumname=mediumfile, 
                        mediumbound=mediumbound, 
                        method=method, verbose=False)

# Create varmed the list of variable medium based on experimental file
varmed = {}
for i in range(X.shape[0]):
    varmed[i] = []
    for j in range(X.shape[1]):
        if parameter.levmed[j] > 1 and X[i,j] > 0:
            varmed[i].append(parameter.medium[j])
varmed = list(varmed.values())

# Get a Cobra training set constrained by varmed
for i in range(X.shape[0]): 
    parameter.get(sample_size=size_i, varmed=varmed[i], verbose=True) 

# Saving file
trainingfile  = DIRECTORY+'Dataset_model/'+mediumname+'_'+parameter.mediumbound + "_Roozbeh_test" 
parameter.save(trainingfile, reduce=reduce)

# Verifying
parameter = TrainingSet()
parameter.load(trainingfile)
print(trainingfile)
parameter.printout()

sample: 0
pass (varmed, obj): ['EX_gal_e_i'] 0.09339486031638583
primal objectif = ['BIOMASS_Ec_iML1515_core_75p37M'] pFBA 0.09339486031638485
sample: 1
pass (varmed, obj): ['EX_gal_e_i'] 0.09864436780342009
primal objectif = ['BIOMASS_Ec_iML1515_core_75p37M'] pFBA 0.09864436780342009
sample: 2
pass (varmed, obj): ['EX_gal_e_i'] 0.13974434548912149
primal objectif = ['BIOMASS_Ec_iML1515_core_75p37M'] pFBA 0.13974434548912149
sample: 3
pass (varmed, obj): ['EX_gal_e_i'] 0.10975073487276607
primal objectif = ['BIOMASS_Ec_iML1515_core_75p37M'] pFBA 0.10975073487276607
sample: 4
pass (varmed, obj): ['EX_gal_e_i'] 0.16057328719817257
primal objectif = ['BIOMASS_Ec_iML1515_core_75p37M'] pFBA 0.16057328719817257
sample: 5
pass (varmed, obj): ['EX_gal_e_i'] 0.16390591787162329
primal objectif = ['BIOMASS_Ec_iML1515_core_75p37M'] pFBA 0.16390591787162329
sample: 6
pass (varmed, obj): ['EX_gal_e_i'] 0.11058389303794224
primal objectif = ['BIOMASS_Ec_iML1515_core_75p37M'] pFBA 0.11058389303794224

In [11]:
import shutil
import numpy as np

from Library.Build_Model import Neural_Model, TrainingSet, MM_LP, Loss_Vout, Loss_SV, Loss_Vin, Loss_Vpos
# Run Mechanistic model (no training) QP (quadratic program) or LP (linear program)
# using E. coli core simulation training sets and EB (or UB) bounds

# What you can change
seed = 10
np.random.seed(seed=seed)  
cobraname =  'iML1515_duplicated' # name of the model 
mediumname = 'iML1515' # name of the medium file 
mediumbound = 'UB' # Exact bound (EB) or upper bound (UB)
expname = 'iML1515_EXP' # name of the experimental dataset for constraints
method = 'pFBA' # FBA, pFBA or EXP
size, size_i  = 110, 10 # expname training set size, training set size per item in expname
reduce = True # Set at True if you want to reduce the model
verbose = True

trainname = mediumname+'_'+mediumbound + "_Roozbeh_test"
timestep =  500 # int(1.0e4) # LP 1.0e4 QP 1.0e5
learn_rate = 1e-3 # LP 0.3 QP 1.0
# End of What you can change

# Create model and run GD for X and Y randomly drawn from trainingfile
trainingfile = DIRECTORY+'Dataset_model/'+trainname

# Copy the model without size
# remove the size part if present
new_train_name = "_".join(trainname.split('_')[:-1])
trainingfile = './Dataset_model/'+new_train_name
# copy the model and paste with new name
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".npz", trainingfile + ".npz")
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".xml", trainingfile + ".xml")

print(f"Training file used: {trainingfile}")

biomass_reaction_name = 'BIOMASS_Ec_iML1515_core_75p37M'
# Get the target index for biomass reaction
original_parameter = TrainingSet()
original_parameter.load(trainingfile)
target_index = [r.id for r in original_parameter.model.reactions].index(biomass_reaction_name)
size = original_parameter.Y.shape[0]


model = Neural_Model(trainingfile = trainingfile, 
              objective=[biomass_reaction_name],#['BIOMASS_Ecoli_core_w_GAM'], 
              model_type = 'MM_LP', 
              timestep = timestep, 
              learn_rate = learn_rate,               
              verbose=True)


Training file used: ./Dataset_model/iML1515_UB_Roozbeh
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\iML1515_UB_Roozbeh_test.xml: True
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\iML1515_UB_Roozbeh_test.xml: True
number of reactions:  544 544
number of metabolites:  1079
filtered measurements size:  1


In [12]:
# Select a random subset of the training set (of specified size)
# With LP we also have to change b_ext and b_int accordingly
ID = np.random.choice(model.X.shape[0], size, replace=False)
model.X, model.Y = model.X[ID,:], model.Y[ID,:]
if model.mediumbound == 'UB':
    model.b_ext = model.b_ext[ID,:]
if model.mediumbound == 'EB':
    model.b_int = model.b_int[ID,:]

# Prints a summary of the model before running
model.printout()
# # Runs the appropriate method
Ypred, Stats = MM_LP(model, verbose=True)
# # Printing results
printout(Ypred, Stats, model)
# Ypred = MM_LP(model, verbose=True, get_stats=False)



training file: ./Dataset_model/iML1515_UB_Roozbeh
model type: MM_LP
model scaler: 0.0
model input dim: 38
model output dim: 1
model medium bound: UB
timestep: 500
training set size (1100, 38) (1100, 1)
LP-Loss 1 0.035345558 0.04389072
LP-Loss 10 80396.57 55.953625
LP-Loss 100 93595640000000.0 12348370000.0


c:\Users\rh2310\projects\amn_release\.env\lib\site-packages\numpy\core\_methods.py:243: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
c:\Users\rh2310\projects\amn_release\.env\lib\site-packages\numpy\core\_methods.py:232: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


AMN output shapes for PoutV, SV, PinV, Vpos, V, outputs (1100, 1) (1100, 1) (1100, 1) (1100, 1) (1100, 544) (1100, 1092)
R2 = -9090201851220925079775834839456562590341624358737175787620391714816.00 (+/- 0.00) Constraint = nan (+/- nan)
Loss Targets nan
Loss SV nan
Loss Vin bound nan
Loss V positive nan


In [16]:
import shutil
import numpy as np

from Library.Build_Model import Neural_Model, TrainingSet, MM_QP, Loss_Vout, Loss_SV, Loss_Vin, Loss_Vpos
# Run Mechanistic model (no training) QP (quadratic program) or LP (linear program)
# using E. coli core simulation training sets and EB (or UB) bounds

# What you can change
seed = 10
np.random.seed(seed=seed)  
cobraname =  'iML1515_duplicated' # name of the model 
mediumname = 'iML1515' # name of the medium file 
mediumbound = 'UB' # Exact bound (EB) or upper bound (UB)
expname = 'iML1515_EXP' # name of the experimental dataset for constraints
method = 'pFBA' # FBA, pFBA or EXP
size, size_i  = 110, 10 # expname training set size, training set size per item in expname
reduce = True # Set at True if you want to reduce the model
verbose = True

trainname = mediumname+'_'+mediumbound + "_Roozbeh_test"
timestep =  500 # int(1.0e4) # LP 1.0e4 QP 1.0e5
learn_rate = 1.0 # LP 0.3 QP 1.0
# End of What you can change

# Create model and run GD for X and Y randomly drawn from trainingfile
trainingfile = DIRECTORY+'Dataset_model/'+trainname

# Copy the model without size
# remove the size part if present
new_train_name = "_".join(trainname.split('_')[:-1])
trainingfile = './Dataset_model/'+new_train_name
# copy the model and paste with new name
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".npz", trainingfile + ".npz")
shutil.copyfile(DIRECTORY+'Dataset_model/'+trainname + ".xml", trainingfile + ".xml")

print(f"Training file used: {trainingfile}")

biomass_reaction_name = 'BIOMASS_Ec_iML1515_core_75p37M'
# Get the target index for biomass reaction
original_parameter = TrainingSet()
original_parameter.load(trainingfile)
target_index = [r.id for r in original_parameter.model.reactions].index(biomass_reaction_name)
size = original_parameter.Y.shape[0]


model = Neural_Model(trainingfile = trainingfile, 
              objective=[biomass_reaction_name],#['BIOMASS_Ecoli_core_w_GAM'], 
              model_type = 'MM_QP', 
              timestep = timestep, 
              learn_rate = learn_rate,               
              verbose=True)


Training file used: ./Dataset_model/iML1515_UB_Roozbeh
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\iML1515_UB_Roozbeh_test.xml: True
Loading model from: c:\Users\rh2310\projects\amn_release\Lab_Notebooks\AMN\..\..\Dataset_model\iML1515_UB_Roozbeh_test.xml: True
number of reactions:  544 544
number of metabolites:  1079
filtered measurements size:  1


In [17]:
# Select a random subset of the training set (of specified size)
# With LP we also have to change b_ext and b_int accordingly
ID = np.random.choice(model.X.shape[0], size, replace=False)
model.X, model.Y = model.X[ID,:], model.Y[ID,:]
if model.mediumbound == 'UB':
    model.b_ext = model.b_ext[ID,:]
if model.mediumbound == 'EB':
    model.b_int = model.b_int[ID,:]

# Prints a summary of the model before running
model.printout()
# # Runs the appropriate method
Ypred, Stats = MM_QP(model, verbose=True)
# # Printing results
printout(Ypred, Stats, model)
# Ypred = MM_LP(model, verbose=True, get_stats=False)



training file: ./Dataset_model/iML1515_UB_Roozbeh
model type: MM_QP
model scaler: 0.0
model input dim: 38
model output dim: 1
model medium bound: UB
timestep: 500
training set size (1100, 38) (1100, 1)
QP-Loss 1 0.035345558 0.04389072
QP-Loss 10 0.009388204 0.011620215
QP-Loss 100 0.0006854391 0.0008010085
AMN output shapes for PoutV, SV, PinV, Vpos, V, outputs (1100, 1) (1100, 1) (1100, 1) (1100, 1) (1100, 544) (1100, 1092)
R2 = 1.00 (+/- 0.00) Constraint = 0.00 (+/- 0.00)
Loss Targets 0.0019878554
Loss SV 0.03188129
Loss Vin bound 1.5454125e-12
Loss V positive 0.0022835152
